# Fault injector (library)

The real game will not be clean: pieces get knocked, dropped, put on the wrong square, a lamp
is switched on, someone bumps the camera. This notebook turns each fault type in
`gambit_bringup/config/faults.yaml` into concrete world changes (set_pose requests for pieces or
the camera, light_config requests), so `run_fault_tests.ipynb` can check that perception + game
logic **notice** and ask the human instead of acting on a wrong board.

**Deliverable:** robustness tests for the "Gazebo vanishing pieces to test game logic" step of the
Updated plan (and the README's safety goal: never move on a board it is unsure about).

**TODO checklist**
- [ ] fault sampling by probability, seeded
- [ ] knocked piece, dropped piece, wrong square, lighting change, camera shift -> requests
- [ ] expected board after each fault (what the vision *should* report)
- [ ] backend: send the requests (reuse teleport's backends, light_config for lights)
- [ ] `undo(fault)`: restore the clean world after the check
- [ ] decide the pass criteria per fault in run_fault_tests (flagged? correct FEN? no robot move?)

In [ ]:
%run ./data_generator.ipynb

In [ ]:
from dataclasses import dataclass, field

FAULTS_PATH = REPO / "gambit_bringup/config/faults.yaml"
with open(FAULTS_PATH) as f:
    FAULTS = yaml.safe_load(f)

## A fault

`requests` are SetPose (pieces / camera) and `lights` are light dicts for `light_config_command`.
`expected_fen` is the board placement a *perfect* observer would report after the fault (e.g. a
dropped piece is gone, a wrong-square piece is on the wrong square); `None` = unchanged.

In [ ]:
@dataclass
class Fault:
    kind: str
    description: str
    requests: list = field(default_factory=list)
    lights: list = field(default_factory=list)
    expected_fen: str | None = None
    meta: dict = field(default_factory=dict)

def sample_fault_kind(rng, cfg=FAULTS):
    # Design: weighted choice over cfg['types'] by their `probability` (normalize so they need not sum to 1).
    raise NotImplementedError

def _pick_piece(tracker, rng, cfg=FAULTS):
    # Design: random occupied square from tracker.on_square, skipping kings when protect_kings
    # (a missing king makes python-chess positions invalid); return (square, instance name).
    raise NotImplementedError

def _uniform(rng, lo_hi):
    # Design: float uniform sample from a [lo, hi] pair in the config.
    raise NotImplementedError

def tilt_quat(tilt_deg, heading, rng=None):
    """Quaternion tilting a piece by `tilt_deg` toward `heading` (rad, in the xy plane)."""
    # Design: axis-angle -> quaternion; the axis is horizontal and perpendicular to the heading
    # direction, so the top of the piece leans toward `heading`. Return (x, y, z, w), unit norm.
    raise NotImplementedError

## Fault generators

Each takes the current `PieceTracker` (the clean position) and an RNG and returns a `Fault`.
They do **not** change the tracker: the tracker stays the ground truth of the *intended* game.

In [ ]:
def knocked_piece(tracker, rng, cfg=FAULTS):
    # Design:
    # - Pick a piece; offset it from its square center by a random distance (offset_m range) in a
    #   random direction, and tilt it (tilt_deg range) the same way.
    # - expected_fen stays None: it is still on its square (but the vision may be less confident).
    # - meta: square, offset, tilt (useful to analyse which offsets the classifier survives).
    raise NotImplementedError

def dropped_piece(tracker, rng, cfg=FAULTS, board=BOARD):
    # Design:
    # - Pick a piece; with on_board_probability it lands somewhere random on the playing area,
    #   otherwise just outside one side (off_board_margin_m) on the frame or the table (z differs).
    # - Lying on its side: z raised by the piece radius, tilted 90 deg in a random heading.
    # - expected_fen = the board with that piece removed (a perfect observer sees it missing).
    raise NotImplementedError

def wrong_square(tracker, rng, cfg=FAULTS):
    """Move a piece to a neighbouring square (as if the last move landed one square off)."""
    # Design:
    # - Candidates = squares within max_distance_squares (king-distance) on the board; drop
    #   occupied ones unless allow_occupied.
    # - No candidate (boxed in) -> degrade to a knocked piece rather than failing.
    # - expected_fen = board with the piece moved there (python-chess copy + set/remove_piece_at).
    raise NotImplementedError

def lighting_change(tracker, rng, cfg=FAULTS):
    # Design: scale the default sun intensity (intensity_scale range, clamp to 1) and jitter its
    # direction (direction_jitter_deg, reuse _rotate_dir); result goes in `lights`, no requests.
    raise NotImplementedError

def camera_shift(tracker, rng, cfg=FAULTS, board=BOARD):
    # Design: random 3D translation of a magnitude in position_m and a random small rotation of a
    # magnitude in rotation_deg added to camera_pose(); one camera_set_pose request. The board
    # itself is unchanged, so expected_fen None; the test is whether the homography recovers.
    raise NotImplementedError

GENERATORS = {"knocked_piece": knocked_piece, "dropped_piece": dropped_piece, "wrong_square": wrong_square,
              "lighting_change": lighting_change, "camera_shift": camera_shift}

def sample_fault(tracker, rng, cfg=FAULTS):
    # Design: sample a kind, dispatch through GENERATORS.
    raise NotImplementedError

def fault_schedule(n_plies, rng, cfg=FAULTS):
    """Plies after which to inject a fault: ~faults_per_game, at least min_plies_between apart."""
    # Design: per ply, inject with probability faults_per_game / n_plies, but never within
    # min_plies_between of the previous fault. Trade-off: simple, so the count is only approximate.
    raise NotImplementedError

## Applying and undoing (TODO)

In [ ]:
def inject(fault, backend):
    """TODO: backend.send_all(fault.requests); for each light run light_config_command (gz CLI)
    or the ROS bridge. Wait capture.settle_s before grabbing a frame."""
    raise NotImplementedError("TODO: inject")

def undo(fault, tracker, backend):
    """TODO: restore the clean world: tracker.requests_for_all() for pieces, camera_set_pose(
    camera_pose()) for the camera, and the default sun from chess_world.sdf for lights."""
    raise NotImplementedError("TODO: undo")

## Self-check

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - seeded rng (FAULTS['seed']), tracker after 1.e4 e5 2.Nf3; sample 200 faults:
#   - every fault has requests or lights; every kind appears at least once
#   - knocked_piece: xy offset from the square center within offset_m
#   - piece faults never move a king (no 'K_' in the request name)
#   - if expected_fen is set it differs from the clean placement
#   - every orientation quaternion has unit norm
#   - the tracker is untouched afterwards (same board_fen, check_consistent())
# - wrong_square with max distance 1 -> chess.square_distance(from, to) == 1
# - fault_schedule(80, rng): consecutive plies at least min_plies_between apart
# Example shape of a test:
# f = knocked_piece(t, rng); assert f.expected_fen is None and len(f.requests) == 1
print("Self-check: not written yet (see plan above)")